# Random Forest with scikit-learn

A Random Forest is an ensemble of decision trees trained with bootstrap
samples and random feature subsets. scikit-learn combines their class
probabilities to predict a label. This notebook uses its library model,
not a custom forest implementation.

Install dependencies with `python -m pip install -r requirements.txt`,
select the project's `.venv` Python kernel, and run cells from top to bottom.
The data cell requires internet access to UCI. This notebook is self-contained:
it copies the preparation from `decision-tree.ipynb` without reading or executing it.
Only training and held-out predictions are included; evaluation is deferred.


## 1. Load the real dataset

Fetch Cleveland data from official UCI Heart Disease dataset 45. Combine its
13 numeric features and `num` target, rename `num` to `target`, drop rows with
any missing value, and place the target last. Keep the original classes 0–4;
do not group them into binary labels. As in the Decision Tree data cell,
use a stratified 80/20 train/test split with `random_state=55`.
This assumes aligned, unique feature/target indices and sufficient rows per class.
The matrices retain a header; model inputs exclude both that header and the target.
Keep `y_test` separate for future evaluation only.


In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)
df = df.rename(columns={'num': 'target'})
df = df.dropna(axis=0, how='any')
df = df[[column for column in df.columns if column != 'target'] + ['target']]

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['target'], random_state=55
)

columns = df.columns.tolist()
training_matrix = [columns] + train_df.to_numpy().tolist()
test_matrix = [columns] + test_df.to_numpy().tolist()
X_train = [row[:-1] for row in training_matrix[1:]]
y_train = [row[-1] for row in training_matrix[1:]]
X_test = [row[:-1] for row in test_matrix[1:]]
y_test = [row[-1] for row in test_matrix[1:]]


## 2. Train on training rows only

The cleaned features are numeric, so no categorical encoder is needed.
Fit the forest only on `X_train` and `y_train`, never on held-out rows.
Use 100 trees and model seed 42 for reproducible runs in the same environment;
the separate dataset split uses seed 55.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


## 3. Predict the held-out rows

Reuse the fitted forest without retraining. Each output names the input
features and predicted `target` class (0–4); no scores are computed.


In [ ]:
predictions = model.predict(X_test)
for row, prediction in zip(X_test, predictions):
    features = dict(zip(columns[:-1], row))
    print(f"{features} -> Predicted target: {prediction}")
